# Action Masking in Gymnasium Taxi-v4

**Project:** Maskable PPO  
**Role:** Action Masking

## 1. Introduction

Action masking is a technique used in reinforcement learning to prevent an agent from selecting invalid actions.

In Gymnasium Taxi-v4, the agent can perform six actions: South, North, East, West, Pickup, and Dropoff. However, the validity of these actions depends on the current environment state.

Maskable PPO uses a binary action mask to exclude invalid actions from the policy's action distribution.

- `1` indicates a valid action.
- `0` indicates an invalid action.

This notebook demonstrates how to retrieve, inspect, and validate action masks in Taxi-v4 and how to integrate them with Maskable PPO

In [9]:
import gymnasium as gym
import numpy as np

from sb3_contrib.common.maskable.utils import get_action_masks

ACTION_NAMES = [
    "South",
    "North",
    "East",
    "West",
    "Pickup",
    "Dropoff",
]

env = gym.make("Taxi-v4")

state, info = env.reset(seed=42)

print("Environment: Taxi-v4")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial state:", state)
print("Initial action mask:", info["action_mask"])

Environment: Taxi-v4
Observation space: Discrete(500)
Action space: Discrete(6)
Initial state: 386
Initial action mask: [1 1 0 1 0 0]


## 2. Retrieving Action Masks

Taxi-v4 provides a state-dependent action mask through its environment API.

We define a function that retrieves the current mask directly from the underlying environment. This function can later be used with `ActionMasker` from `sb3-contrib`.

In [10]:
def mask_fn(env):
    """Return the action mask for the current Taxi-v4 state"""
    return env.unwrapped.action_mask(env.unwrapped.s)


mask = mask_fn(env)

print("Current state:", env.unwrapped.s)
print("Action mask:", mask)

for action_id, action_name in enumerate(ACTION_NAMES):
    status = "Valid" if mask[action_id] else "Invalid"
    print(f"{action_id}: {action_name:<8} → {status}")

Current state: 386
Action mask: [1 1 0 1 0 0]
0: South    → Valid
1: North    → Valid
2: East     → Invalid
3: West     → Valid
4: Pickup   → Invalid
5: Dropoff  → Invalid


## 3. Testing Action Masks in Different States

Action masks in Taxi-v4 are state-dependent: the set of valid actions changes depending on the taxi's position, passenger location, and destination.

We test four representative situations:
1. The taxi is away from the passenger.
2. The taxi is at the passenger's pickup location.
3. The passenger is inside the taxi.
4. The taxi is at the destination with the passenger inside.

For each situation, we display the state, its decoded components, the action mask, and the available actions

In [11]:
LOCATIONS = {
    0: (0, 0),  # Red
    1: (0, 4),  # Green
    2: (4, 0),  # Yellow
    3: (4, 3),  # Blue
}

def inspect_state(taxi_row, taxi_col, passenger_loc, destination):
    """Inspect the action mask for a specific Taxi-v4 state"""

    state = env.unwrapped.encode(
        taxi_row,
        taxi_col,
        passenger_loc,
        destination
    )

    mask = env.unwrapped.action_mask(state)

    available_actions = [
        ACTION_NAMES[i]
        for i, is_valid in enumerate(mask)
        if is_valid
    ]

    print(f"State: {state}")
    print(f"Taxi position: ({taxi_row}, {taxi_col})")
    print(f"Passenger location: {passenger_loc}")
    print(f"Destination: {destination}")
    print(f"Action mask: {mask}")
    print(f"Available actions: {available_actions}")
    print("-" * 55)

    return mask


print("CASE 1: Taxi away from passenger")
mask_1 = inspect_state(2, 2, 0, 1)

print("CASE 2: Taxi at passenger pickup location")
mask_2 = inspect_state(0, 0, 0, 1)

print("CASE 3: Passenger inside taxi")
mask_3 = inspect_state(2, 2, 4, 1)

print("CASE 4: Taxi at destination with passenger")
mask_4 = inspect_state(0, 4, 4, 1)

CASE 1: Taxi away from passenger
State: 241
Taxi position: (2, 2)
Passenger location: 0
Destination: 1
Action mask: [1 1 1 1 0 0]
Available actions: ['South', 'North', 'East', 'West']
-------------------------------------------------------
CASE 2: Taxi at passenger pickup location
State: 1
Taxi position: (0, 0)
Passenger location: 0
Destination: 1
Action mask: [1 0 1 0 1 0]
Available actions: ['South', 'East', 'Pickup']
-------------------------------------------------------
CASE 3: Passenger inside taxi
State: 257
Taxi position: (2, 2)
Passenger location: 4
Destination: 1
Action mask: [1 1 1 1 0 0]
Available actions: ['South', 'North', 'East', 'West']
-------------------------------------------------------
CASE 4: Taxi at destination with passenger
State: 97
Taxi position: (0, 4)
Passenger location: 4
Destination: 1
Action mask: [1 0 0 1 0 1]
Available actions: ['South', 'West', 'Dropoff']
-------------------------------------------------------


## 4. Validating Action Masks

We validate the masks for four representative Taxi-v4 states.

The tests verify that pickup and dropoff actions are available only in appropriate situations and that every mask contains six binary values

In [12]:
# Check that masks have the correct shape and values
for mask in [mask_1, mask_2, mask_3, mask_4]:
    assert mask.shape == (6,), "Action mask must contain 6 values"
    assert np.all(np.isin(mask, [0, 1])), "Mask must be binary"
    assert np.any(mask == 1), "At least one action must be valid"

# Check pickup and dropoff rules
assert mask_1[4] == 0, "Pickup must be invalid away from passenger"
assert mask_2[4] == 1, "Pickup must be valid at passenger location"
assert mask_3[4] == 0, "Pickup must be invalid when passenger is inside taxi"
assert mask_4[5] == 1, "Dropoff must be valid at destination"

print("All action masking tests passed!")

All action masking tests passed!


## 5. Integration with Maskable PPO

Maskable PPO needs access to the current action mask during action selection.

We use the `ActionMasker` wrapper from `sb3-contrib` to expose the mask through the standard masking interface.

This allows Maskable PPO to retrieve valid actions automatically while interacting with Taxi-v4.

In [13]:
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import (
    get_action_masks,
    is_masking_supported,
)

# Create a separate environment for integration testing
masked_env = ActionMasker(
    gym.make("Taxi-v4"),
    mask_fn
)

obs, info = masked_env.reset(seed=42)

print("Masking supported:", is_masking_supported(masked_env))
print("Observation:", obs)
print("Action mask:", get_action_masks(masked_env))

Masking supported: True
Observation: 386
Action mask: [1 1 0 1 0 0]


## 6. Testing Masked Action Selection

We create an untrained Maskable PPO model to verify that action masks are respected during action selection.

The goal is not to evaluate performance, but to confirm that invalid actions are excluded when the policy selects an action

In [14]:
from sb3_contrib import MaskablePPO

model = MaskablePPO(
    "MlpPolicy",
    masked_env,
    verbose=0,
    seed=42,
)

obs, info = masked_env.reset(seed=42)

mask = get_action_masks(masked_env)

selected_actions = []

for _ in range(100):
    action, _ = model.predict(
        obs,
        action_masks=mask,
        deterministic=False,
    )

    action = int(action)
    selected_actions.append(action)

    assert mask[action] == 1, (
        f"Invalid action selected: {ACTION_NAMES[action]}"
    )

print("Current state:", obs)
print("Action mask:", mask)
print("Selected actions:", sorted(set(selected_actions)))
print("All 100 selected actions were valid!")

Current state: 386
Action mask: [1 1 0 1 0 0]
Selected actions: [0, 1, 3]
All 100 selected actions were valid!
